# 04 - Exports and Interactive Map

In [1]:
import sys
sys.path.insert(0, '..')

import json
import shutil

import folium
import geopandas as gpd
import pandas as pd
from shapely import set_precision

from src.features import compute_mapillary_url
from src.score import INSUFFICIENT_DATA_TIER

reliable = gpd.read_parquet('../data/processed/_reliable_scored.parquet')
reliable = gpd.GeoDataFrame(reliable, geometry='geometry', crs='EPSG:4326')
low_confidence = gpd.read_parquet('../data/processed/_low_confidence_final.parquet')
low_confidence = gpd.GeoDataFrame(low_confidence, geometry='geometry', crs='EPSG:4326')
with open('../data/processed/_sensitivity_summary.json') as f:
    sens_summary = json.load(f)

## Export `segments_scored.geojson` / `.gpkg` and `summary_statistics.csv`

In [2]:
reliable.to_file('../outputs/segments_scored.geojson', driver='GeoJSON')
reliable.to_file('../outputs/segments_scored.gpkg', driver='GPKG', layer='segments_scored')
print(f'Saved segments_scored.geojson/.gpkg ({len(reliable)} segments)')

total = len(reliable) + len(low_confidence)
tier_counts = reliable['risk_tier'].value_counts()
tier_counts[INSUFFICIENT_DATA_TIER] = len(low_confidence)

rows = [
    {'metric': 'total_segments', 'value': total},
    {'metric': 'reliable_segments', 'value': len(reliable)},
    {'metric': 'low_confidence_segments', 'value': len(low_confidence)},
]
for tier, count in tier_counts.items():
    key = tier.replace(' ', '_')
    rows.append({'metric': f'risk_tier_count_{key}', 'value': count})
    rows.append({'metric': f'risk_tier_pct_{key}', 'value': round(count / total * 100, 2)})
rows += [
    {'metric': 'mean_speed_safety_score', 'value': round(reliable['speed_safety_score'].mean(), 2)},
    {'metric': 'max_speed_safety_score', 'value': reliable['speed_safety_score'].max()},
    {'metric': 'mean_speed_gap_kmh', 'value': round(reliable['speed_gap'].mean(), 2)},
    {'metric': 'correlation_speed_safety_score_vs_ranked_percentile', 'value': round(sens_summary['correlation_with_ranked_percentile'], 4)},
    {'metric': 'sensitivity_avg_top20pct_overlap_pct', 'value': round(sens_summary['average_overlap_pct'], 2)},
]
summary_df = pd.DataFrame(rows)
summary_df.to_csv('../outputs/summary_statistics.csv', index=False)
display(summary_df)

Saved segments_scored.geojson/.gpkg (14546 segments)


,metric,value
0,total_segments,69966.000
1,reliable_segments,14546.000
2,low_confidence_segments,55420.000
3,risk_tier_count_Low_risk,14164.000
4,risk_tier_pct_Low_risk,20.240
5,risk_tier_count_Medium_risk,382.000
6,risk_tier_pct_Medium_risk,0.550
7,risk_tier_count_Insufficient_data,55420.000
8,risk_tier_pct_Insufficient_data,79.210
9,mean_speed_safety_score,22.020


## Build the interactive Folium map

Reliable (scored) segments get full tooltips and a clickable Mapillary popup. Low-confidence segments are shown as a separate, lighter-weight, off-by-default layer (coarser geometry simplification, minimal tooltip, no popup) -- embedding all ~70k segments at full fidelity produced an unusably large (60MB+) HTML file, so the low-confidence layer trades some visual fidelity for a browser-friendly file size.

In [3]:
RISK_COLORS = {
    'High risk': '#E24B4A', 'Medium risk': '#EF9F27',
    'Low risk': '#1D9E75', INSUFFICIENT_DATA_TIER: '#B4B2A9',
}
RELIABLE_TOOLTIP_FIELDS = ['segment_id', 'road_class', 'SpeedLimit', 'F85thPercentileSpeed',
                           'speed_gap', 'risk_tier', 'speed_safety_score', 'RoadLength_km', 'Sample_Size_Total']
RELIABLE_TOOLTIP_ALIASES = ['DISSOLVE_ID:', 'Class:', 'Speed limit (km/h):', '85th pct speed (km/h):',
                            'Speed gap (km/h):', 'Risk tier:', 'Speed Safety Score:', 'Road length (km):', 'Sample size:']
LOW_CONF_TOOLTIP_FIELDS = ['segment_id', 'road_class', 'risk_tier', 'RoadLength_km', 'Sample_Size_Total']
LOW_CONF_TOOLTIP_ALIASES = ['DISSOLVE_ID:', 'Class:', 'Risk tier:', 'Road length (km):', 'Sample size:']

In [4]:
def _prep_reliable(reliable):
    cols = RELIABLE_TOOLTIP_FIELDS + ['country', 'StreetImageLink', 'geometry']
    out = reliable[cols].copy()
    out['geometry'] = out.geometry.simplify(0.0003, preserve_topology=True)
    out['geometry'] = gpd.GeoSeries(set_precision(out.geometry.values, grid_size=0.00001), crs='EPSG:4326')
    for c in ['SpeedLimit', 'F85thPercentileSpeed', 'speed_gap', 'RoadLength_km']:
        out[c] = pd.to_numeric(out[c], errors='coerce').round(1)
    out['speed_safety_score'] = pd.to_numeric(out['speed_safety_score'], errors='coerce').round(1)
    out['Sample_Size_Total'] = pd.to_numeric(out['Sample_Size_Total'], errors='coerce').round(0)
    out = compute_mapillary_url(out)
    out['popup'] = out['mapillary_url'].map(
        lambda u: f'<a href="{u}" target="_blank" rel="noopener">View street imagery</a>' if pd.notna(u) else '')
    return out.drop(columns=['mapillary_url', 'StreetImageLink'])

def _prep_low_confidence(low_confidence):
    low_confidence = low_confidence.copy()
    low_confidence['risk_tier'] = INSUFFICIENT_DATA_TIER
    cols = LOW_CONF_TOOLTIP_FIELDS + ['country', 'geometry']
    out = low_confidence[cols].copy()
    out['geometry'] = out.geometry.simplify(0.0015, preserve_topology=True)
    out['geometry'] = gpd.GeoSeries(set_precision(out.geometry.values, grid_size=0.0001), crs='EPSG:4326')
    out['RoadLength_km'] = pd.to_numeric(out['RoadLength_km'], errors='coerce').round(1)
    out['Sample_Size_Total'] = pd.to_numeric(out['Sample_Size_Total'], errors='coerce').round(0)
    return out

reliable_map = _prep_reliable(reliable)
low_conf_map = _prep_low_confidence(low_confidence)

In [5]:
all_bounds = pd.concat([reliable_map.geometry, low_conf_map.geometry]).total_bounds
center = [(all_bounds[1] + all_bounds[3]) / 2, (all_bounds[0] + all_bounds[2]) / 2]
m = folium.Map(location=center, zoom_start=5, tiles='CartoDB Positron')

title_html = '''
<div style="position: fixed; top: 12px; right: 12px; z-index: 9999; background: white;
            padding: 8px 14px; border-radius: 4px; box-shadow: 0 1px 4px rgba(0,0,0,0.4);
            font-size: 16px; font-weight: 600;">
  AI for Safer Roads &mdash; Speed Safety Score
</div>
'''
m.get_root().html.add_child(folium.Element(title_html))

legend_html = '''
<div style="position: fixed; bottom: 24px; left: 24px; z-index: 9999; background: white;
            padding: 10px 14px; border-radius: 4px; box-shadow: 0 1px 4px rgba(0,0,0,0.4); font-size: 13px;">
  <b>Risk tier</b><br>
  <span style="display:inline-block;width:12px;height:12px;background:#E24B4A;margin-right:6px;"></span>High risk<br>
  <span style="display:inline-block;width:12px;height:12px;background:#EF9F27;margin-right:6px;"></span>Medium risk<br>
  <span style="display:inline-block;width:12px;height:12px;background:#1D9E75;margin-right:6px;"></span>Low risk<br>
  <span style="display:inline-block;width:12px;height:12px;background:#B4B2A9;margin-right:6px;"></span>Insufficient data
</div>
'''
m.get_root().html.add_child(folium.Element(legend_html))

def style_function(feature):
    tier = feature['properties']['risk_tier']
    return {'color': RISK_COLORS.get(tier, '#B4B2A9'), 'weight': 4 if tier == 'High risk' else 2, 'opacity': 0.85}

In [6]:
countries = sorted(set(reliable_map['country']) | set(low_conf_map['country']))

for country in countries:
    rel_sub = reliable_map[reliable_map['country'] == country]
    fg_rel = folium.FeatureGroup(name=f'{country} — scored segments', show=True)
    folium.GeoJson(
        rel_sub.__geo_interface__, style_function=style_function,
        tooltip=folium.GeoJsonTooltip(fields=RELIABLE_TOOLTIP_FIELDS, aliases=RELIABLE_TOOLTIP_ALIASES, sticky=True),
        popup=folium.GeoJsonPopup(fields=['popup'], aliases=[''], labels=False, parse_html=True),
    ).add_to(fg_rel)
    fg_rel.add_to(m)

    lc_sub = low_conf_map[low_conf_map['country'] == country]
    fg_lc = folium.FeatureGroup(name=f'{country} — insufficient data', show=False)
    folium.GeoJson(
        lc_sub.__geo_interface__, style_function=style_function,
        tooltip=folium.GeoJsonTooltip(fields=LOW_CONF_TOOLTIP_FIELDS, aliases=LOW_CONF_TOOLTIP_ALIASES, sticky=True),
    ).add_to(fg_lc)
    fg_lc.add_to(m)

folium.LayerControl(collapsed=False).add_to(m)

m.save('../outputs/map.html')
shutil.copy('../outputs/map.html', '../docs/index.html')
print('Saved outputs/map.html and copied it to docs/index.html')
print('Countries on map:', countries)
print(f'Reliable (scored) features plotted: {len(reliable_map)}')
print(f'Low-confidence (insufficient data) features plotted: {len(low_conf_map)}')

Saved outputs/map.html and copied it to docs/index.html
Countries on map: ['India', 'Thailand']
Reliable (scored) features plotted: 14546
Low-confidence (insufficient data) features plotted: 55420


In [7]:
print('Map saved to outputs/map.html and docs/index.html — open in a browser to view (not rendered inline to keep this notebook small).')

Map saved to outputs/map.html and docs/index.html — open in a browser to view (not rendered inline to keep this notebook small).
